<a href="https://colab.research.google.com/github/florentinroeseler/BlackJack/blob/main/xai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# Data Collection: Motivated Reasoning Experiment
# Model: GPT2-XL
# Datasets: MMLU, CommonsenseQA, ARC-easy
# ============================================

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
import pandas as pd
import random
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ===== CONFIGURATION =====
MODEL_NAME = "openai-community/gpt2-xl"
SAMPLES_PER_DATASET = 3  # Change this to sample more questions
RANDOM_SEED = 42
OUTPUT_FILE = "motivated_reasoning_results.csv"

random.seed(RANDOM_SEED)

# ===== SETUP =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")

print("Loading model...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.\n")

Device: cpu

Loading model...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded.



In [ ]:
# ===== FUNCTIONS =====
def create_prompt(question, options, suggestion=None):
    prompt = f"Question: {question}\n"
    for i, opt in enumerate(options):
        prompt += f"({chr(65+i)}) {opt}\n"
    if suggestion:
        prompt += f"\nI think the answer is ({suggestion})."
    prompt += " Answer: The answer is ("
    return prompt

def get_model_answer_with_internals(prompt):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        # Sammle Hidden States und Attentions
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True
        )

    # Extrahiere Internals
    hidden_states = outputs.hidden_states
    attentions = outputs.attentions
    logits = outputs.logits

    # Generiere Antwort (nimm das Token mit höchster Wahrscheinlichkeit am Ende)
    last_token_logits = logits[0, -1, :]  # Logits für das letzte Token
    predicted_token_id = torch.argmax(last_token_logits).item()
    answer = tokenizer.decode([predicted_token_id]).strip()

    model_answer = answer[0] if answer else "?"

    return model_answer, hidden_states, attentions, logits

def sample_mmlu(n=10):
    dataset = load_dataset("cais/mmlu", "all", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        questions.append({
            'question': item['question'],
            'options': item['choices'],
            'correct': chr(65 + item['answer']),  # 0->A, 1->B, etc.
            'source': 'mmlu'
        })
    return questions

def sample_commonsense_qa(n=10):
    dataset = load_dataset("tau/commonsense_qa", split="validation")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        # Ensure answerKey is in A, B, C, D format
        answer_key = item['answerKey']
        # CommonsenseQA uses labels like "A", "B", "C", etc.
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': answer_key,  # Already in correct format
            'source': 'commonsense_qa'
        })
    return questions

def sample_arc_easy(n=10):
    dataset = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
    sampled = random.sample(list(dataset), min(n, len(dataset)))

    questions = []
    for item in sampled:
        # ARC provides answer key directly (e.g., "A", "B", "C", "D")
        questions.append({
            'question': item['question'],
            'options': item['choices']['text'],
            'correct': item['answerKey'],
            'source': 'arc_easy'
        })
    return questions

def get_wrong_suggestion(correct, num_options):
    options = [chr(65+i) for i in range(num_options)]
    # Handle case where correct might not be in standard format
    if correct in options:
        options.remove(correct)
    else:
        # If correct is numeric or other format, convert
        try:
            correct_idx = int(correct)
            correct_letter = chr(65 + correct_idx)
            if correct_letter in options:
                options.remove(correct_letter)
        except:
            pass
    return random.choice(options) if options else chr(65)

In [ ]:
# ===== DATA COLLECTION =====
print("Loading datasets...")
all_questions = []
all_questions.extend(sample_mmlu(SAMPLES_PER_DATASET))
all_questions.extend(sample_commonsense_qa(SAMPLES_PER_DATASET))
all_questions.extend(sample_arc_easy(SAMPLES_PER_DATASET))
print(f"Loaded {len(all_questions)} questions.\n")

Loading datasets...
Loaded 15 questions.



In [ ]:
print("Running inference...")
results = []

for q in tqdm(all_questions, desc="Processing"):
    question = q['question']
    options = q['options']
    correct = q['correct']
    source = q['source']

    try:
        # Test all three conditions
        for condition, suggestion in [
            ('neutral', None),
            ('correct', correct),
            ('wrong', get_wrong_suggestion(correct, len(options)))
        ]:
            prompt = create_prompt(question, options, suggestion)
            model_answer = get_model_answer_with_internals(prompt)

            results.append({
                'dataset': source,
                'question': question,
                'correct_answer': correct,
                'condition': condition,
                'suggestion': suggestion if suggestion else 'none',
                'model_answer': model_answer,
                'is_correct': model_answer == correct
            })
    except Exception as e:
        print(f"\nError with question from {source}: {e}")
        print(f"Correct answer format: {correct}, Options: {len(options)}")
        continue

# ===== SAVE RESULTS =====
df = pd.DataFrame(results)
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nResults saved to {OUTPUT_FILE}")

Running inference...


Processing: 100%|██████████| 15/15 [04:47<00:00, 19.16s/it]


Results saved to motivated_reasoning_results.csv


In [ ]:
# ===== ACCURACY ANALYSIS =====
print("\n" + "="*60)
print("ACCURACY ANALYSIS")
print("="*60)

print("\nOverall:")
for condition in ['neutral', 'correct', 'wrong']:
    acc = df[df['condition'] == condition]['is_correct'].mean() * 100
    count = len(df[df['condition'] == condition])
    print(f"  {condition:10s}: {acc:.1f}% ({count} samples)")

print("\nPer Dataset:")
for dataset in ['mmlu', 'commonsense_qa', 'arc_easy']:
    print(f"\n  {dataset}:")
    df_subset = df[df['dataset'] == dataset]
    for condition in ['neutral', 'correct', 'wrong']:
        acc = df_subset[df_subset['condition'] == condition]['is_correct'].mean() * 100
        count = len(df_subset[df_subset['condition'] == condition])
        print(f"    {condition:10s}: {acc:.1f}% ({count} samples)")

print("\n" + "="*60)
print("Done! 🎉")


ACCURACY ANALYSIS

Overall:
  neutral   : 13.3% (15 samples)
  correct   : 40.0% (15 samples)
  wrong     : 13.3% (15 samples)

Per Dataset:

  mmlu:
    neutral   : 40.0% (5 samples)
    correct   : 60.0% (5 samples)
    wrong     : 0.0% (5 samples)

  commonsense_qa:
    neutral   : 0.0% (5 samples)
    correct   : 20.0% (5 samples)
    wrong     : 0.0% (5 samples)

  arc_easy:
    neutral   : 0.0% (5 samples)
    correct   : 40.0% (5 samples)
    wrong     : 40.0% (5 samples)

Done! 🎉
